# Hands-On 7: Build a ReAct-Style Agent

ReAct = Reason + Act. The agent thinks, acts (calls tools), observes, and repeats until it has a final answer.

#### Imports

We import `tool` for tools, `ChatOpenAI` for the model, `MessagesState` for message state, and `ToolNode` for executing tool calls.

In [1]:
from typing import Literal

import os
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage

In [2]:
load_dotenv(r"D:\Agent_Flow\.env", override=True)

print("OpenRouter API key loaded:", bool(os.getenv("OPENROUTER_API_KEY")))

OpenRouter API key loaded: True


#### Define Tools

The agent uses these tools multiple times in a loop: get weather, calculate math, and get current time.

In [3]:
@tool
def get_weather(city: str) -> str:
    """Get the current temperature of a city."""
    
    weather = {
        "new york": "22°C, clear",
        "mumbai": "30°C, humid",
        "tokyo": "18°C, rainy"
    }
    
    return weather.get(
        city.lower(),
        f"No weather data available for {city}"
    )

In [4]:
@tool
def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression."""
    
    allowed = set("0123456789+-*/(). ")
    
    if not set(expression).issubset(allowed):
        return "Error: invalid expression"
    
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

In [5]:
@tool
def get_current_time() -> str:
    """Get the current system time."""
    
    from datetime import datetime
    
    return datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

In [6]:
print("Weather:", get_weather.invoke("Mumbai"))
print("Calculation:", calculate.invoke("7 * 6 + 2"))
print("Current time:", get_current_time.invoke(""))

Weather: 30°C, humid
Calculation: 44
Current time: 2026-08-21 00:21:56


#### Configure Model

We bind all tools to the LLM so it can request tool calls during its reasoning.

In [7]:
tools = [
    get_weather,
    calculate,
    get_current_time
]

print("Tools:", [t.name for t in tools])

Tools: ['get_weather', 'calculate', 'get_current_time']


In [8]:
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

print("OpenRouter LLM configured successfully")

OpenRouter LLM configured successfully


In [9]:
llm_with_tools = llm.bind_tools(tools)

print("Tools successfully bound to model")

Tools successfully bound to model


#### Build Agent Graph

The ReAct loop: model → tools → model → ... until the model produces a final answer with no tool calls.

In [10]:
def agent_node(state: MessagesState):
    response = llm_with_tools.invoke(
        state["messages"]
    )
    
    return {
        "messages": [response]
    }

## Routing Logic

The model decides whether it needs a tool.

In [11]:
def should_continue(
    state: MessagesState
) -> Literal["tools", "__end__"]:
    
    last_message = state["messages"][-1]
    
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    
    return "__end__"

### Create ToolNode

In [12]:
tool_node = ToolNode(tools)

print("ToolNode created successfully")

ToolNode created successfully


## Build Graph

In [13]:
graph = StateGraph(MessagesState)

graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)

graph.add_edge(
    START,
    "agent"
)

graph.add_conditional_edges(
    "agent",
    should_continue
)

graph.add_edge(
    "tools",
    "agent"
)

print("ReAct graph created successfully")

ReAct graph created successfully


### Compile Agent

In [14]:
react_agent = graph.compile()

print("ReAct agent compiled successfully")

ReAct agent compiled successfully


#### Test: Weather Query

Invoke the agent with a weather question to see it call the tool and return the observation.

In [15]:
result = react_agent.invoke({
    "messages": [
        HumanMessage(
            content="What's the weather in Mumbai?"
        )
    ]
})

print("=== Full Trace ===")

for msg in result["messages"]:
    
    if msg.type == "human":
        print(f"👤 User: {msg.content}")
    
    elif msg.type == "ai":
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(
                    f"🤖 AI → Tool: "
                    f"{tc['name']}({tc['args']})"
                )
        else:
            print(f"✅ AI: {msg.content}")
    
    elif msg.type == "tool":
        print(f"🔧 Tool Result: {msg.content}")

=== Full Trace ===
👤 User: What's the weather in Mumbai?
🤖 AI → Tool: get_weather({'city': 'Mumbai'})
🔧 Tool Result: 30°C, humid
✅ AI: The weather in Mumbai is currently 30°C and humid.


#### Test: Math Calculation

Invoke with a math expression to verify the calculator tool works.

In [16]:
result = react_agent.invoke({
    "messages": [
        HumanMessage(
            content="Calculate 7 * 6 + 2"
        )
    ]
})

for msg in result["messages"]:
    
    if msg.type == "human":
        print(f"👤 User: {msg.content}")
    
    elif msg.type == "ai":
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(
                    f"🤖 AI → Tool: "
                    f"{tc['name']}({tc['args']})"
                )
        else:
            print(f"✅ AI: {msg.content}")
    
    elif msg.type == "tool":
        print(f"🔧 Tool Result: {msg.content}")

👤 User: Calculate 7 * 6 + 2
🤖 AI → Tool: calculate({'expression': '7 * 6 + 2'})
🔧 Tool Result: 44
✅ AI: The result of the calculation \( 7 \times 6 + 2 \) is 44.


#### Test: Multiple Tool Calls in One Query

The agent may call several tools before answering a compound query.

In [17]:
result = react_agent.invoke({
    "messages": [
        HumanMessage(
            content=(
                "What is the weather in New York, "
                "what time is it, and calculate 100 / 4?"
            )
        )
    ]
})

print("=== ReAct Trace ===")

for msg in result["messages"]:
    
    if msg.type == "human":
        print(f"👤 User: {msg.content}")
    
    elif msg.type == "ai":
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(
                    f"🤖 AI → Tool: "
                    f"{tc['name']}({tc['args']})"
                )
        else:
            print(f"✅ AI: {msg.content}")
    
    elif msg.type == "tool":
        print(f"🔧 Tool Result: {msg.content}")

=== ReAct Trace ===
👤 User: What is the weather in New York, what time is it, and calculate 100 / 4?
🤖 AI → Tool: get_weather({'city': 'New York'})
🤖 AI → Tool: get_current_time({})
🤖 AI → Tool: calculate({'expression': '100 / 4'})
🔧 Tool Result: 22°C, clear
🔧 Tool Result: 2026-08-21 00:26:17
🔧 Tool Result: 25.0
✅ AI: The current weather in New York is 22°C and clear. The time is 00:26:17. Additionally, the result of the calculation \(100 / 4\) is 25.0.


#### Test: Query Without Tools

No tool calls are needed for a simple knowledge question, so the agent answers directly.

In [18]:
result = react_agent.invoke({
    "messages": [
        HumanMessage(
            content="What is the capital of France?"
        )
    ]
})

print("Final answer:")
print(result["messages"][-1].content)

Final answer:
The capital of France is Paris.


In [19]:
result = react_agent.invoke({
    "messages": [
        HumanMessage(
            content=(
                "Calculate (45 * 8) / 4, "
                "tell me the weather in Tokyo, "
                "and tell me the current time."
            )
        )
    ]
})

print("=== Complex Query Trace ===")

for msg in result["messages"]:
    
    if msg.type == "human":
        print(f"👤 User: {msg.content}")
    
    elif msg.type == "ai":
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(
                    f"🤖 AI → Tool: "
                    f"{tc['name']}({tc['args']})"
                )
        else:
            print(f"✅ AI: {msg.content}")
    
    elif msg.type == "tool":
        print(f"🔧 Tool Result: {msg.content}")

=== Complex Query Trace ===
👤 User: Calculate (45 * 8) / 4, tell me the weather in Tokyo, and tell me the current time.
🤖 AI → Tool: calculate({'expression': '(45 * 8) / 4'})
🤖 AI → Tool: get_weather({'city': 'Tokyo'})
🤖 AI → Tool: get_current_time({})
🔧 Tool Result: 90.0
🔧 Tool Result: 18°C, rainy
🔧 Tool Result: 2026-08-21 00:27:11
✅ AI: The result of the calculation \((45 * 8) / 4\) is **90**. 

The current weather in Tokyo is **18°C** and rainy. 

The current time is **00:27:11**.


In [20]:
print("=== HANDS-ON 7 COMPLETE ===")

print("✓ Tools defined")
print("✓ OpenRouter model configured")
print("✓ Tools bound to model")
print("✓ ReAct agent graph created")
print("✓ Model → Tool routing implemented")
print("✓ Tool → Model loop implemented")
print("✓ Multiple tool calls tested")
print("✓ No-tool query tested")
print("✓ Agent stops when final answer is ready")

=== HANDS-ON 7 COMPLETE ===
✓ Tools defined
✓ OpenRouter model configured
✓ Tools bound to model
✓ ReAct agent graph created
✓ Model → Tool routing implemented
✓ Tool → Model loop implemented
✓ Multiple tool calls tested
✓ No-tool query tested
✓ Agent stops when final answer is ready


#### Summary

The ReAct agent loop: reason → act (tool call) → observe (tool result) → repeat until final answer.